# Speech-to-Text Pipeline — Graduation Project

**Model:** OpenAI Whisper (open-source, 100% free)  
**Task:** Extract speech from uploaded audio → feed to your summary model API  
**Handles:** Background noise, mid-audio silences, overlapping speech, accents

---
### Pipeline Overview
```
Audio File Upload
      ↓
Audio Preprocessing (noise reduction + normalization)
      ↓
Whisper STT (transcription with timestamps)
      ↓
Post-processing (clean text + smart merging of segments)
      ↓
Summary Model API Integration
```

##  Step 1 — Install Dependencies

In [18]:
!pip install -q openai-whisper
!pip install -q noisereduce
!pip install -q pydub
!pip install -q librosa
!pip install -q soundfile
!pip install -q ffmpeg-python
!apt-get install -y -q ffmpeg   

print("All dependencies installed.")

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 141 not upgraded.
All dependencies installed.


In [19]:
import os
from google.colab import files as colab_files  


KAGGLE_AUDIO_PATH = "/kaggle/input/datasets/rusulaltukhi/lectures-records/1. Introduction and the geometric viewpoint on physics..mp3"  


IS_KAGGLE = os.path.exists("/kaggle")

if IS_KAGGLE:
    AUDIO_PATH = KAGGLE_AUDIO_PATH
    print(f" Kaggle mode — using: {AUDIO_PATH}")
    if not os.path.exists(AUDIO_PATH):
        print("File not found. Please upload your audio via the Kaggle 'Add Data' panel and update KAGGLE_AUDIO_PATH above.")
else:
    print("Colab mode — please upload your audio file:")
    uploaded = colab_files.upload()
    AUDIO_PATH = list(uploaded.keys())[0]
    print(f" Uploaded: {AUDIO_PATH}")

 Kaggle mode — using: /kaggle/input/datasets/rusulaltukhi/lectures-records/1. Introduction and the geometric viewpoint on physics..mp3


## Audio Preprocessing

This step:
- Converts any format → WAV (16kHz mono) which Whisper expects
- Applies **noise reduction** using spectral gating
- Normalizes volume levels so quiet speech is amplified properly

In [20]:
import numpy as np
import librosa
import soundfile as sf
import noisereduce as nr
from pydub import AudioSegment
import warnings
warnings.filterwarnings("ignore")


SAMPLE_RATE     = 16000 
PROCESSED_PATH  = "/tmp/processed_audio.wav"
NOISE_REDUCTION = True   


def preprocess_audio(input_path: str, output_path: str, apply_noise_reduction: bool = True) -> str:

    print(f" Loading audio: {input_path}")
    

    ext = os.path.splitext(input_path)[1].lower().lstrip(".")
    if ext != "wav":
        print(f"   Converting {ext.upper()} → WAV ...")
        temp_wav = "/tmp/temp_input.wav"
        audio_seg = AudioSegment.from_file(input_path)
        audio_seg.export(temp_wav, format="wav")
        load_path = temp_wav
    else:
        load_path = input_path


    print(f"   Resampling to {SAMPLE_RATE} Hz mono ...")
    audio, sr = librosa.load(load_path, sr=SAMPLE_RATE, mono=True)
    duration = len(audio) / sr
    print(f"   Duration: {duration:.1f} seconds")

   
    if apply_noise_reduction:
 
     
        noise_sample_duration = min(0.5, duration * 0.1)  
        noise_sample = audio[: int(noise_sample_duration * sr)]
        audio = nr.reduce_noise(
            y=audio,
            sr=sr,
            y_noise=noise_sample,
            stationary=False,   
            prop_decrease=0.75  
        )

   
    print("   Normalizing amplitude ...")
    max_val = np.max(np.abs(audio))
    if max_val > 0:
        audio = audio / max_val * 0.95 


    sf.write(output_path, audio, sr)
    print(f" Preprocessed audio saved → {output_path}")
    return output_path



processed_audio = preprocess_audio(AUDIO_PATH, PROCESSED_PATH, NOISE_REDUCTION)

 Loading audio: /kaggle/input/datasets/rusulaltukhi/lectures-records/1. Introduction and the geometric viewpoint on physics..mp3
   Converting MP3 → WAV ...
   Resampling to 16000 Hz mono ...
   Duration: 4108.3 seconds
   Normalizing amplitude ...
 Preprocessed audio saved → /tmp/processed_audio.wav


**Whisper Model**

Whisper model size options (all free/open-source):

| Model   | VRAM  | Speed    | Accuracy  | Best for                        |
|---------|-------|----------|-----------|---------------------------------|
| `tiny`  | ~1GB  | Fastest  | Basic     | Quick tests                     |
| `base`  | ~1GB  | Fast     | Good      | Short/clean audio               |
| `small` | ~2GB  | Medium   | Better    | General use                     |
| `medium`| ~5GB  | Slower   | Great     | Noisy/accented audio |
| `large` | ~10GB | Slowest  | Best      | Maximum accuracy (Kaggle GPU P100 can handle this) |



In [21]:
import whisper
import torch


WHISPER_MODEL_SIZE = "medium" 


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Running on: {device.upper()}")
if device == "cpu":
    print("  No GPU detected. Transcription will be slower. Enable GPU in Kaggle: Settings → Accelerator → GPU")

print(f" Loading Whisper '{WHISPER_MODEL_SIZE}' model (downloading if first time) ...")
model = whisper.load_model(WHISPER_MODEL_SIZE, device=device)
print(f" Whisper '{WHISPER_MODEL_SIZE}' model loaded on {device.upper()}")

 Running on: CUDA
 Loading Whisper 'medium' model (downloading if first time) ...
 Whisper 'medium' model loaded on CUDA


## Transcribe Audio

In [22]:
import json
import re


LANGUAGE = "en"          
TASK     = "transcribe"  
BEAM_SIZE = 5             
TEMPERATURE = 0.0        


print(" Transcribing audio with Whisper ...")
print("   (This may take a moment depending on audio length and model size)\n")

result = model.transcribe(
    processed_audio,
    language=LANGUAGE,
    task=TASK,
    beam_size=BEAM_SIZE,
    temperature=TEMPERATURE,
    word_timestamps=True,          
    condition_on_previous_text=True,  
    no_speech_threshold=0.6,       
    logprob_threshold=-1.0,      
    compression_ratio_threshold=2.4,
    verbose=False
)

raw_text    = result["text"].strip()
segments    = result["segments"]
detected_lang = result.get("language", "en")

print(f" Transcription complete!")
print(f"   Detected language : {detected_lang}")
print(f"   Total segments    : {len(segments)}")
print(f"   Raw text length   : {len(raw_text)} characters")

 Transcribing audio with Whisper ...
   (This may take a moment depending on audio length and model size)



100%|██████████| 410828/410828 [11:20<00:00, 603.88frames/s] 

 Transcription complete!
   Detected language : en
   Total segments    : 1014
   Raw text length   : 43408 characters


## Post-Processing

Cleans and structures the raw Whisper output:
- Removes filler words (`um`, `uh`, etc.)
- Merges fragmented segments separated by silences/pauses
- Generates a timestamped transcript and a clean plain-text version

In [23]:
def format_timestamp(seconds: float) -> str:
    """Convert float seconds → HH:MM:SS format."""
    hours   = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs    = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"


def clean_text(text: str, remove_fillers: bool = True) -> str:

    if remove_fillers:
        fillers = r"\b(um+|uh+|hmm+|ah+|er+|erm+|like um|you know|i mean)\b"
        text = re.sub(fillers, "", text, flags=re.IGNORECASE)
    

    text = re.sub(r" {2,}", " ", text)   # Multiple spaces → single space
    text = re.sub(r" ([.,!?;:])", r"\1", text)  # Remove space before punctuation
    text = re.sub(r"\n{3,}", "\n\n", text)  # Max two consecutive newlines
    return text.strip()


def postprocess_segments(segments: list, silence_gap: float = 2.0) -> dict:

    timestamped = []
    paragraphs  = []
    current_paragraph = []
    prev_end = 0.0

    for seg in segments:
        start = seg["start"]
        end   = seg["end"]
        text  = seg["text"].strip()

        # Skip empty or very short noise segments
        if len(text) < 2:
            continue

     
        timestamped.append({
            "start": format_timestamp(start),
            "end":   format_timestamp(end),
            "start_sec": round(start, 2),
            "end_sec":   round(end, 2),
            "text": text
        })

      
        gap = start - prev_end
        if gap >= silence_gap and current_paragraph:
            paragraphs.append(" ".join(current_paragraph))
            current_paragraph = []
        
        current_paragraph.append(text)
        prev_end = end


    if current_paragraph:
        paragraphs.append(" ".join(current_paragraph))

    
    merged = "\n\n".join(paragraphs)
    final  = clean_text(merged)

    return {
        "timestamped_transcript": timestamped,
        "paragraph_text":         merged,
        "clean_text":             final,
        "total_segments":         len(timestamped),
        "total_paragraphs":       len(paragraphs)
    }



output = postprocess_segments(segments, silence_gap=2.0)

print(" Post-processing complete!")
print(f"   Segments   : {output['total_segments']}")
print(f"   Paragraphs : {output['total_paragraphs']}")

 Post-processing complete!
   Segments   : 1014
   Paragraphs : 121


## View Results

In [24]:
print("=" * 60)
print(" TIMESTAMPED TRANSCRIPT")
print("=" * 60)
for seg in output["timestamped_transcript"]:
    print(f"[{seg['start']} → {seg['end']}]  {seg['text']}")

print()
print("=" * 60)
print(" CLEAN TEXT (ready for summary model)")
print("=" * 60)
print(output["clean_text"])

 TIMESTAMPED TRANSCRIPT
[00:00:10 → 00:00:14]  The key textbooks for this class is Sean Carroll's textbook
[00:00:14 → 00:00:15]  on general relativity.
[00:00:19 → 00:00:21]  It's now almost 20 years old.
[00:00:22 → 00:00:25]  I would say, I think it's listed on the website as required.
[00:00:25 → 00:00:27]  I would actually call it sort of semi-required.
[00:00:28 → 00:00:34]  It is where I will tend to post most of the readings
[00:00:34 → 00:00:34]  to the course.
[00:00:35 → 00:00:39]  It is a really good, complete textbook
[00:00:39 → 00:00:43]  for a one-semester course, which is what we have.
[00:00:43 → 00:00:45]  And I will not be going through the entire thing.
[00:00:45 → 00:00:47]  You can't in a one-semester course.
[00:00:48 → 00:00:49]  And I will, from time to time, there
[00:00:49 → 00:00:53]  will be a few topics that I cannot go into as deeply
[00:00:53 → 00:00:53]  as I would like.
[00:00:54 → 00:00:56]  If we had two semesters, maybe I'd
[00:00:56 → 00:00:56]  b

## Save Transcription Outputs

In [25]:
import json

OUTPUT_DIR = "/kaggle/working" if IS_KAGGLE else "/tmp"


json_path = os.path.join(OUTPUT_DIR, "transcription_full.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump({
        "source_file":            os.path.basename(AUDIO_PATH),
        "whisper_model":          WHISPER_MODEL_SIZE,
        "detected_language":      detected_lang,
        "total_segments":         output["total_segments"],
        "total_paragraphs":       output["total_paragraphs"],
        "clean_text":             output["clean_text"],
        "timestamped_transcript": output["timestamped_transcript"]
    }, f, indent=2, ensure_ascii=False)


txt_path = os.path.join(OUTPUT_DIR, "clean_transcript.txt")
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(output["clean_text"])

print(f" Saved:")
print(f"  Full JSON    → {json_path}")
print(f"  Clean text   → {txt_path}")

 Saved:
  Full JSON    → /kaggle/working/transcription_full.json
  Clean text   → /kaggle/working/clean_transcript.txt


## Pipeline Summary Report

In [27]:
import librosa


original_audio, sr = librosa.load(AUDIO_PATH, sr=None, mono=True)
duration_secs = len(original_audio) / sr
duration_str  = format_timestamp(duration_secs)

word_count  = len(output["clean_text"].split())
char_count  = len(output["clean_text"])

print("=" * 60)
print(" PIPELINE SUMMARY REPORT")
print("=" * 60)
print(f"  Audio file      : {os.path.basename(AUDIO_PATH)}")
print(f"  Duration        : {duration_str}")
print(f"  Whisper model   : {WHISPER_MODEL_SIZE}")
print(f"  Device          : {device.upper()}")
print(f"  Language        : {detected_lang}")
print(f"  Noise reduction : {'Yes' if NOISE_REDUCTION else 'No'}")
print()
print(f"  Segments found  : {output['total_segments']}")
print(f"  Paragraphs      : {output['total_paragraphs']}")
print(f"  Word count      : {word_count}")
print(f"  Character count : {char_count}")
print()
print(f"  Output JSON     : {json_path}")
print(f"  Output TXT      : {txt_path}")
print("=" * 60)

[src/libmpg123/id3.c:process_comment():587] error: No comment text / valid description?


 PIPELINE SUMMARY REPORT
  Audio file      : 1. Introduction and the geometric viewpoint on physics..mp3
  Duration        : 01:08:28
  Whisper model   : medium
  Device          : CUDA
  Language        : en
  Noise reduction : Yes

  Segments found  : 1014
  Paragraphs      : 121
  Word count      : 8033
  Character count : 43447

  Output JSON     : /kaggle/working/transcription_full.json
  Output TXT      : /kaggle/working/clean_transcript.txt
